# 03 – Proposed method: two-step MODIS–ECOSTRESS fusion for night LST

**Step 1, block correction (980 m).** A random forest learns where the MODIS night pattern differs from the ECOSTRESS block means (sensor, time gap, coast, elevation) and corrects the MODIS block values.
**Step 2, within-block anomaly (70 m).** A second random forest learns how much each 70 m pixel deviates from its block mean; the re-centred anomaly is added to the corrected block value.

Compared on the same clear nights, the same folds and the same ECOSTRESS reference:

| Method | Inputs |
|---|---|
| MODIS upsampled | MODIS only |
| TsHARP (Agam et al. 2007) | MODIS + Sentinel-2 NDVI |
| Sentinel-2-only random forest | MODIS + Sentinel-2 NDVI and NDBI |
| 70 m anomaly (step 2 only) | MODIS + all 70 m predictors |
| Block correction (step 1 only) | MODIS pattern + block-mean predictors + night context |
| **Proposed fusion** (steps 1 + 2) | all of the above |

Validation: 6-fold leave-nights-out; paired bootstrap over nights; ablation of the block correction; within-block skill at 70, 140 and 490 m.

Outputs: `data/results_*.csv`, `figures/03_*.png`.

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/KeremBasimc/istanbul-night-heat-island"
if "google.colab" in sys.modules or Path("/content").exists():
    ROOT = Path("/content/istanbul-night-heat-island")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
FIGS = ROOT / "figures"
DATA.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)

import earthaccess
# Tries, in order: environment variables, ~/.netrc, then an interactive prompt.
print("Earthdata login ok:", earthaccess.login().authenticated)

## Clear nights and grids

In [ ]:
import time
import numpy as np
import pandas as pd
from nhi import data, downscale as ds, features as fe, model as M

nights_csv = DATA / "ecostress_nights.csv"
if nights_csv.exists():
    per_night = pd.read_csv(nights_csv, parse_dates=["local"])
else:
    g = data.search_ecostress("2018-07-01", "2025-09-30")
    g = data.add_clear_fraction(g[g.is_night], DATA / "ecostress")
    per_night = data.best_granule_per_night(g)
    per_night.to_csv(nights_csv, index=False)
clear = per_night[per_night.clear_frac >= 0.7].reset_index(drop=True)
coarse, fine = data.grids()
print(len(clear), "clear nights")

## Static predictor layers (cached per year)

In [ ]:
t0 = time.time()
wc = fe.worldcover_fractions(fine)
sea_km = fe.distance_to_sea(wc["water"]) / 1000
elev = fe.elevation(fine)
land = wc["water"] < 0.5

def static_for(year):
    path = DATA / f"static_{year}.npz"
    if path.exists():
        return dict(np.load(path))
    s2 = fe.sentinel2_indices(year, fine)
    layers = {"ndvi": s2["ndvi"], "ndbi": s2["ndbi"],
              "tree": wc["tree"], "grass": wc["grass"], "shrub": wc["shrub"], "crop": wc["crop"],
              "built": wc["built"], "bare": wc["bare"],
              "sea_dist_km": sea_km, "elev": elev,
              "landsat_day": fe.landsat_day_lst(year, fine)}
    np.savez_compressed(path, **layers)
    return layers

years = sorted(pd.to_datetime(clear.night_date).dt.year.unique())
STATIC = {y: static_for(y) for y in years}
bldg_h = fe.ghsl_building_height(fine, folder=DATA / "ghsl")   # GHSL 2018, same for all years
for y in years:
    STATIC[y]["bldg_height"] = bldg_h
STATIC_DEV = {y: M.static_with_deviations(STATIC[y]) for y in years}
print("static layers for", years, f"in {time.time() - t0:.0f} s")

## Assemble every clear night, screen gross sensor disagreement, score the non-learning benchmarks

In [ ]:
t0 = time.time()
NIGHTS = []
for r in clear.itertuples():
    paths = data.download([r.url_LST, r.url_cloud, r.url_water], DATA / "ecostress")
    eco = np.where(land, data.ecostress_on_grid(*paths, fine).values, np.nan)   # same land mask for all methods
    passes = data.load_modis_night_passes(pd.Timestamp(r.night_date).date(), coarse, hdf_folder=DATA / "modis_hdf")
    pick = data.closest_modis_pass(passes, pd.Timestamp(r.local))
    if pick is None:
        continue
    platform, mp, gap = pick
    modis_c = mp["lst"].values.astype("float32")
    year = pd.Timestamp(r.night_date).year
    NIGHTS.append({"key": str(r.night_date), "layers": M.night_layers(STATIC_DEV[year], modis_c, pd.Timestamp(r.local), gap),
                   "y": M.anomaly_target(eco), "eco": eco, "modis_coarse": modis_c, "land": land,
                   "platform": platform, "gap_h": gap, "local": r.local})
print(len(NIGHTS), f"nights assembled in {time.time() - t0:.0f} s")

# Cloud that one of the two masks missed shows up as gross disagreement at 980 m.
NIGHTS, dropped = M.screen_nights(NIGHTS, max_median_diff=5.0, min_r=0.0)
print("dropped by the screen:"); print(dropped.to_string(index=False))
print(len(NIGHTS), "nights used")

bench = []
for n in NIGHTS:
    ndvi_land = np.where(land, n["layers"]["ndvi"], np.nan)
    for name, pred in [("MODIS upsampled", np.where(land, ds.upsample(n["modis_coarse"]), np.nan)),
                       ("TsHARP", ds.tsharp(n["modis_coarse"], ndvi_land)[0])]:
        m = ds.metrics(pred, n["eco"]); m.update(night_date=n["key"], method=name)
        bench.append(m)

## Where is the error? Between 980 m blocks or inside them?

Downscaling can only add detail inside a block. If most of the disagreement with ECOSTRESS sits between blocks, the MODIS block values themselves need correcting.

In [ ]:
def crmse(a, b):
    v = np.isfinite(a) & np.isfinite(b); d = a[v] - b[v]
    return float(np.sqrt(np.mean((d - d.mean()) ** 2)))

dec = pd.DataFrame([{"night_date": n["key"],
                     "between_blocks": crmse(n["modis_coarse"], ds.block_mean(n["eco"])),
                     "within_blocks_sd": float(np.nanstd(n["y"]))} for n in NIGHTS])
dec.describe().loc[["mean", "50%"]].round(3)

## Main comparison: 6-fold leave-nights-out

Every method is scored on the same nights with the same folds. The proposed fusion is reported with its two steps separately.

In [ ]:
t0 = time.time()
FINE = M.features_for()
BLOCK = M.block_columns()
fusion, skill, IMPS = M.cross_validate_fusion(NIGHTS, FINE, BLOCK, n_folds=6)
print(f"fusion: {time.time() - t0:.0f} s"); t0 = time.time()
s2only = M.cross_validate(NIGHTS, M.S2_ONLY, n_folds=6).assign(method="S2-only RF")
print(f"S2-only RF: {time.time() - t0:.0f} s")

meta = pd.DataFrame([{"night_date": n["key"], "platform": n["platform"], "gap_h": n["gap_h"],
                      "month": pd.Timestamp(n["local"]).month} for n in NIGHTS])
res = pd.concat([pd.DataFrame(bench), s2only, fusion]).merge(meta, on="night_date")
res.to_csv(DATA / "results_main.csv", index=False)
skill.to_csv(DATA / "results_scale_skill.csv", index=False)
order = ["MODIS upsampled", "TsHARP", "S2-only RF", "70 m anomaly", "Block correction", "Proposed fusion"]
summary = res.groupby("method")[["rmse", "crmse", "mae", "bias", "r"]].agg(["mean", "median"]).loc[order].round(3)
summary

## Paired bootstrap over nights (negative = proposed is better)

In [ ]:
def paired(a, b, col, n_boot=5000, seed=0):
    pa = res[res.method == a].set_index("night_date")[col]
    pb = res[res.method == b].set_index("night_date")[col]
    d = (pa - pb).dropna().values
    boots = np.random.default_rng(seed).choice(d, (n_boot, d.size), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"comparison": f"{a} - {b}", "metric": col, "mean_diff": d.mean(), "ci_low": lo, "ci_high": hi,
            "relative_%": 100 * d.mean() / pb.mean(), "nights_better": int((d < 0).sum()), "nights": d.size}

tests = pd.DataFrame([paired("Proposed fusion", b, c) for b in ("MODIS upsampled", "TsHARP", "S2-only RF")
                      for c in ("crmse", "rmse")])
tests.to_csv(DATA / "results_bootstrap.csv", index=False)
tests.round(3)

## Is the block correction more than smoothing?

The ablation shows that step 1 relies on the MODIS neighbourhood pattern. Fixed spatial filters of MODIS are therefore scored as extra non-learning baselines on the same nights.

In [ ]:
from scipy import ndimage

def nan_smooth(a, size=None, sigma=None):
    ok = np.isfinite(a)
    f = (lambda x: ndimage.uniform_filter(x, size)) if size else (lambda x: ndimage.gaussian_filter(x, sigma))
    s, w = f(np.where(ok, a, 0.0)), f(ok.astype(float))
    return np.where(w > 1e-6, s / np.maximum(w, 1e-9), np.nan)

def bilinear(mc):
    filled = np.where(np.isfinite(mc), mc, nan_smooth(mc, size=3))
    filled = np.where(np.isfinite(filled), filled, np.nanmean(mc))
    return ndimage.zoom(filled, data.FACTOR, order=1)[: mc.shape[0] * data.FACTOR, : mc.shape[1] * data.FACTOR]

rows = []
for n in NIGHTS:
    mc = n["modis_coarse"].astype("float64")
    for name, p in {"MODIS bilinear": bilinear(mc), "MODIS 3x3 mean": ds.upsample(nan_smooth(mc, size=3)),
                    "MODIS 5x5 mean": ds.upsample(nan_smooth(mc, size=5)),
                    "MODIS Gaussian s=1.5": ds.upsample(nan_smooth(mc, sigma=1.5))}.items():
        m = ds.metrics(np.where(land, p, np.nan), n["eco"]); m.update(night_date=n["key"], method=name)
        rows.append(m)
smooth = pd.DataFrame(rows)
smooth.to_csv(DATA / "results_smoothing.csv", index=False)
cmp_ = pd.concat([smooth, res[res.method.isin(["MODIS upsampled", "Block correction", "Proposed fusion"])]])
print(cmp_.groupby("method")[["crmse", "r"]].agg(["mean", "median"]).sort_values(("crmse", "mean")).round(3).to_string())
best = smooth.groupby("method").crmse.mean().idxmin()
res = pd.concat([res, smooth.merge(meta, on="night_date")])
paired("Proposed fusion", best, "crmse")

## At which scale is the night detail predictable?

Correlation between predicted and observed within-block anomalies, after averaging both to 70, 140 and 490 m.

In [ ]:
import matplotlib.pyplot as plt

sk = skill.groupby("method")[[c for c in skill.columns if c.startswith(("r_", "sd_eco"))]].median().round(3)
print(sk.to_string())
fig, ax = plt.subplots(figsize=(5, 3.6), constrained_layout=True)
scales = [70, 140, 490]
for name, mk in (("Proposed", "o-"), ("TsHARP", "s--")):
    ax.plot(scales, [sk.loc[name, f"r_{s}m"] for s in scales], mk, label=name)
ax.set_xscale("log"); ax.set_xticks(scales, [f"{s} m" for s in scales])
ax.set_ylabel("Median r of within-block anomaly"); ax.set_xlabel("Evaluation scale")
ax.grid(alpha=.3); ax.legend()
fig.savefig(FIGS / "03_scale_skill.png", dpi=150)
plt.show()

## Ablation of the block correction (6-fold)

Each predictor group is removed in turn from step 1; the increase in bias-removed RMSE (block correction only) shows what the correction relies on.

In [ ]:
t0 = time.time()
groups = list(M.STATIC_GROUPS)
variants = {"all": M.block_columns()}
for g in groups:
    variants[f"without {g}"] = M.block_columns([x for x in groups if x != g])
variants["without MODIS pattern"] = M.block_columns(modis_pattern=False)
variants["without night context"] = M.block_columns(night_context=False)
abl = []
for name, cols in variants.items():
    r_, _, _ = M.cross_validate_fusion(NIGHTS, FINE, cols, n_folds=6, fine=False, block_trees=100)
    abl.append({"variant": name, "crmse": r_.crmse.mean(), "crmse_median": r_.crmse.median(), "r": r_.r.mean()})
abl = pd.DataFrame(abl)
abl["crmse_increase"] = abl.crmse - abl.loc[abl.variant == "all", "crmse"].item()
abl.to_csv(DATA / "results_ablation.csv", index=False)
print(f"ablation: {time.time() - t0:.0f} s")
abl.round(3)

## Feature importance of the two steps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
IMPS["block"].sort_values().plot.barh(ax=axes[0])
axes[0].set_title("Step 1: what corrects MODIS at 980 m")
fi = IMPS["fine"].groupby(lambda c: c.removesuffix("_dev")).sum().sort_values()
fi.plot.barh(ax=axes[1])
axes[1].set_title("Step 2: what explains the 70 m anomaly")
for ax in axes:
    ax.set_xlabel("Impurity importance")
fig.savefig(FIGS / "03_feature_importance.png", dpi=150)
plt.show()

## Method comparison and an example night

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, col, label in zip(axes, ["crmse", "rmse"], ["Bias-removed RMSE (°C)", "RMSE (°C)"]):
    ax.boxplot([res.loc[res.method == m, col].dropna() for m in order], showfliers=False)
    ax.set_xticks(range(1, len(order) + 1), order, rotation=20)
    ax.set_title(label); ax.grid(axis="y", alpha=.3)
fig.suptitle(f"6-fold leave-nights-out against ECOSTRESS, {len(NIGHTS)} clear nights")
fig.savefig(FIGS / "03_method_comparison.png", dpi=150)
plt.show()

## Example night (median gain, chosen by rule)

Representative example night, chosen by a fixed rule from the cross-validation results: summer (Jun-Sep), at least median land coverage, MODIS gap up to 1.5 h, and a fusion gain over MODIS closest to the median gain of all nights. The model shown is trained on all other nights.

In [ ]:
piv = res.pivot_table(index="night_date", columns="method", values="crmse")
gain = 100 * (piv["Proposed fusion"] - piv["MODIS upsampled"]) / piv["MODIS upsampled"]
by_key = {n["key"]: n for n in NIGHTS}
cover = pd.Series({k: np.isfinite(n["eco"]).sum() for k, n in by_key.items()})
ok = [k for k in gain.dropna().index if k in by_key
      and pd.Timestamp(by_key[k]["local"]).month in (6, 7, 8, 9)
      and by_key[k]["gap_h"] <= 1.5 and cover[k] >= cover.median()]
ex_key = (gain[ok] - gain.median()).abs().idxmin()
ex = by_key[ex_key]
print(f"median gain over MODIS {gain.median():.1f} %; example night {ex_key} gain {gain[ex_key]:.1f} %")

bm, fm = M.fit_fusion([n for n in NIGHTS if n["key"] != ex["key"]], FINE, BLOCK)
pred = M.predict_fusion(bm, fm, ex, FINE, BLOCK)
tsh = ds.tsharp(ex["modis_coarse"], np.where(land, ex["layers"]["ndvi"], np.nan))[0]
vmin, vmax = np.nanpercentile(ex["eco"], [2, 98])
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), constrained_layout=True)
for ax, arr, title in [(axes[0, 0], ex["eco"], "ECOSTRESS 70 m (reference, not used for this night)"),
                       (axes[0, 1], np.where(land, ds.upsample(ex["modis_coarse"]), np.nan), f"MODIS {ex['platform'].title()} 980 m"),
                       (axes[1, 0], tsh, "TsHARP 70 m"),
                       (axes[1, 1], pred, "Proposed fusion 70 m")]:
    im = ax.imshow(arr, cmap="inferno", vmin=vmin, vmax=vmax)
    mt = ds.metrics(arr, ex["eco"]) if arr is not ex["eco"] else None
    ax.set_title(title + (f"\nbias-removed RMSE {mt['crmse']:.2f} °C, r {mt['r']:.2f}" if mt else ""))
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="Land surface temperature (°C)", shrink=0.8)
fig.suptitle(f"Night of {ex['key']} (ECOSTRESS {pd.Timestamp(ex['local']):%H:%M}, MODIS gap {ex['gap_h']:.1f} h); "
             f"median-gain night of {len(gain.dropna())}")
fig.savefig(FIGS / "03_example_night.png", dpi=150)
plt.show()

## Does TsHARP's vegetation rule hold at night?

TsHARP fits LST = a + b * fc between 980 m blocks for each scene and applies it at 70 m. Here the fitted slope b and its R^2 are recorded for every night (MODIS) and compared with the same fit for the daytime Landsat LST of the same year. `r_within` is the correlation between the observed within-block anomaly (ECOSTRESS at night, Landsat by day) and the fc anomaly at 70 m, i.e. whether the rule holds where TsHARP applies it.

In [ ]:
def fc_of(ndvi):
    lo, hi = np.nanpercentile(ndvi, [2, 98])
    return ds.fractional_cover(ndvi, lo, hi)


def coarse_fit(lst_c, fc_c):
    ok = np.isfinite(lst_c) & np.isfinite(fc_c)
    b, a = np.polyfit(fc_c[ok], lst_c[ok], 1)
    return b, np.corrcoef(fc_c[ok], lst_c[ok])[0, 1] ** 2


def within_r(lst_f, fc_f):
    a = lst_f - ds.upsample(ds.block_mean(lst_f))
    f = fc_f - ds.upsample(ds.block_mean(fc_f))
    v = np.isfinite(a) & np.isfinite(f)
    return np.corrcoef(a[v], f[v])[0, 1]


rows = []
for n in NIGHTS:
    fc = fc_of(np.where(land, n["layers"]["ndvi"], np.nan))
    fc_c = ds.block_mean(fc)
    b, r2 = coarse_fit(n["modis_coarse"].astype("float64"), fc_c)
    b_eco, r2_eco = coarse_fit(ds.block_mean(n["eco"]), fc_c)
    rows.append({"scene": n["key"], "kind": "night", "platform": n["platform"],
                 "slope_K_per_fc": b, "r2": r2, "slope_eco_blocks": b_eco, "r2_eco_blocks": r2_eco,
                 "r_within": within_r(n["eco"], fc)})
for y in years:
    fc = fc_of(np.where(land, STATIC[y]["ndvi"], np.nan))
    day = np.where(land, STATIC[y]["landsat_day"], np.nan)
    b, r2 = coarse_fit(ds.block_mean(day), ds.block_mean(fc))
    rows.append({"scene": f"Landsat summer {y}", "kind": "day", "platform": "landsat",
                 "slope_K_per_fc": b, "r2": r2, "r_within": within_r(day, fc)})
slopes = pd.DataFrame(rows)
slopes.to_csv(DATA / "results_tsharp_slopes.csv", index=False)
print(slopes.groupby("kind")[["slope_K_per_fc", "r2", "r_within"]].describe().T.round(3).to_string())
nt = slopes[slopes.kind == "night"]
print(f"night slope negative on {(nt.slope_K_per_fc < 0).sum()}/{len(nt)} nights; "
      f"median R2 night {nt.r2.median():.2f} vs day {slopes[slopes.kind == 'day'].r2.median():.2f}")

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.boxplot([nt.slope_K_per_fc, slopes[slopes.kind == "day"].slope_K_per_fc], showfliers=True)
ax.set_xticks([1, 2], [f"Night, MODIS ({len(nt)})", f"Day, Landsat ({(slopes.kind == 'day').sum()})"])
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("TsHARP slope b (K per unit vegetation cover)")
ax.set_title("The vegetation-temperature rule by day and at night")
fig.savefig(FIGS / "03_tsharp_slopes.png", dpi=150)
plt.show()

## Spatial check: train on one half of the city, test on the other

Leave-nights-out keeps test nights unseen, but every location is seen in training on other nights. Here the fusion is also trained on the western half only and scored on the eastern half (and the reverse), with a 5-block (about 5 km) buffer between them that is used for neither. Same night folds as above.

In [ ]:
ncol = NIGHTS[0]["modis_coarse"].shape[1]
mid, buf = ncol // 2, 2
cols = np.arange(ncol)
halves = {"west": cols < mid - buf, "east": cols > mid + buf}
reg_c = {h: np.broadcast_to(m, NIGHTS[0]["modis_coarse"].shape) for h, m in halves.items()}
reg_f = {h: ds.upsample(m.astype("float32")) > 0.5 for h, m in reg_c.items()}

rng = np.random.default_rng(0)
tables = [M.block_table(n) for n in NIGHTS]
fold_of = np.arange(len(NIGHTS)) % 6


def fit_pair(train_idx, region=None):
    keep_c = reg_c[region].ravel() if region else True
    btr = pd.concat([tables[i][tables[i].ok & keep_c] for i in train_idx])
    bm = M.make_block_model(0, 100).fit(btr[BLOCK].fillna(-999).values, btr["target"].values)
    samples = [M.training_sample(NIGHTS[i]["layers"], FINE, NIGHTS[i]["y"],
                                 NIGHTS[i]["land"] & (reg_f[region] if region else True), 2000, rng)
               for i in train_idx]
    fm = M.make_model(0).fit(np.concatenate([s[0] for s in samples]), np.concatenate([s[1] for s in samples]))
    return bm, fm


t0 = time.time()
# Results are saved after every fold; a rerun resumes from the folds already in the file.
part_csv = DATA / "results_spatial_partial.csv"
rows = pd.read_csv(part_csv).drop(columns="fold", errors="ignore").to_dict("records") if part_csv.exists() else []
done = {r["night_date"] for r in rows}
for f in range(6):
    if all(NIGHTS[i]["key"] in done for i in range(len(NIGHTS)) if fold_of[i] == f):
        print(f"fold {f} already saved")
        continue
    train_idx = [i for i in range(len(NIGHTS)) if fold_of[i] != f]
    test_idx = [i for i in range(len(NIGHTS)) if fold_of[i] == f]
    models = {"all": fit_pair(train_idx), "west": fit_pair(train_idx, "west"), "east": fit_pair(train_idx, "east")}
    for i in test_idx:
        n = NIGHTS[i]
        for test_half, train_half in (("east", "west"), ("west", "east")):
            m_t = reg_f[test_half]
            cands = {"MODIS upsampled": np.where(n["land"], ds.upsample(n["modis_coarse"]), np.nan),
                     "Fusion, trained everywhere": M.predict_fusion(*models["all"], n, FINE, BLOCK),
                     "Fusion, trained on other half": M.predict_fusion(*models[train_half], n, FINE, BLOCK)}
            for name, p in cands.items():
                mt = ds.metrics(np.where(m_t, p, np.nan), n["eco"])
                rows.append({"night_date": n["key"], "test_half": test_half, "method": name, **mt})
    pd.DataFrame(rows).to_csv(part_csv, index=False)
    print(f"fold {f} done, {time.time() - t0:.0f} s")
spatial = pd.DataFrame(rows)
spatial.to_csv(DATA / "results_spatial.csv", index=False)
sp = spatial.pivot_table(index=["night_date", "test_half"], columns="method", values="crmse").dropna()
print(len(sp), "night-halves")
print(sp.groupby(level="test_half").mean().round(3).to_string())
print(sp.mean().round(3).to_string())
for a, b in [("Fusion, trained on other half", "MODIS upsampled"),
             ("Fusion, trained on other half", "Fusion, trained everywhere")]:
    d = (sp[a] - sp[b]).groupby(level="night_date").mean().values
    boots = np.random.default_rng(0).choice(d, (5000, d.size)).mean(axis=1)
    print(f"{a} - {b}: {d.mean():.3f} °C, 95% CI [{np.percentile(boots, 2.5):.3f}, {np.percentile(boots, 97.5):.3f}], "
          f"better on {(d < 0).sum()}/{d.size} nights")

## Notes

- Bias-removed RMSE (`crmse`) is the main score: the night-wide offset between MODIS and ECOSTRESS (time gap, sensor) is the same for every method and is not what downscaling targets.
- The block correction learns where the MODIS pattern differs from ECOSTRESS; its night-wide mean is removed from the target, so it cannot change the night-wide offset.
- All folds split by night, so no pixel of a test night is ever seen in training.